# Day 55: Round-Robin and Least-Connections Load Balancers

**Layer:** Production


## Concept Overview

Implement both round-robin and least-connections load balancers and benchmark them under realistic workloads with variable request durations.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Load Balancer Comparison

Implement both round-robin and least-connections load balancers and benchmark them under realistic w...


In [ ]:
import time, numpy as np

class LoadBalancer:
    def __init__(self, n_workers, strategy):
        self.workers = [{"load": 0, "processed": 0} for _ in range(n_workers)]
        self.strategy = strategy
        self.rr_idx = 0

    def route(self, duration):
        if self.strategy == "round_robin":
            idx = self.rr_idx % len(self.workers)
            self.rr_idx += 1
        elif self.strategy == "least_connections":
            idx = min(range(len(self.workers)), key=lambda i: self.workers[i]["load"])
        self.workers[idx]["load"] += 1
        latency = duration / (1.0 + self.workers[idx]["load"] * 0.01)
        self.workers[idx]["load"] -= 1
        self.workers[idx]["processed"] += 1
        return idx, latency

np.random.seed(42)
n_req = 1000
durations = np.random.lognormal(3, 1, n_req)  # heavy-tailed

for strategy in ["round_robin", "least_connections"]:
    lb = LoadBalancer(n_workers=4, strategy=strategy)
    latencies = [lb.route(d)[1] for d in durations]
    loads = [w["processed"] for w in lb.workers]
    print(f"{strategy}:")
    print(f"  P50={np.percentile(latencies,50):.1f}ms P99={np.percentile(latencies,99):.1f}ms")
    print(f"  Worker loads: {loads} (std={np.std(loads):.0f})")

## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- Implement both round-robin and least-connections load balancers and benchmark them under realistic workloads with variable request durations..
- Least-connections outperforms round-robin for heavy-tailed request distributions because it naturally routes away from currently-busy workers — round-robin ignores the fact that some requests take 10x longer than others..
- Day 55 implementation complete.
